In [1]:
# =========================
# Cell 1: Imports
# =========================
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [2]:
# =========================
# Cell 2: Config
# =========================
@dataclass
class Config:
    data_dir: str = "data"
    reranked_train_path: str = "reranked_train_predictions.json"
    reranked_dev_path: str = "reranked_dev_predictions.json"
    model_name: str = "bert-base-uncased"
    output_dir: str = "outputs/bert_claim_classifier_noise"
    max_length: int = 512
    seed: int = 42
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    train_batch_size: int = 8
    eval_batch_size: int = 8
    num_train_epochs: int = 4
    warmup_ratio: float = 0.1
    evid_sep: str = "[EVID_SEP]"


config = Config()

label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3,
}
id2label = {v: k for k, v in label2id.items()}

In [3]:
# =========================
# Cell 3: Reproducibility
# =========================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(config.seed)

In [4]:
# =========================
# Cell 4: Data Loading
# =========================
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


data_dir = Path(config.data_dir)
train_claims = load_json(data_dir / "train-claims.json")
dev_claims = load_json(data_dir / "dev-claims.json")
evidence_db = load_json(data_dir / "evidence.json")
reranked_train = load_json(Path(config.reranked_train_path))

print(f"Train claims: {len(train_claims)}")
print(f"Dev claims:   {len(dev_claims)}")
print(f"Evidence:     {len(evidence_db)}")
print(f"Reranked train: {len(reranked_train)}")

Train claims: 1228
Dev claims:   154
Evidence:     1208827
Reranked train: 1228


In [5]:
# =========================
# Cell 5: Build Text Pairs
# =========================
def build_text_b(evidence_ids, evidence_db, evid_sep: str) -> str:
    evidence_texts = [evidence_db[eid] for eid in evidence_ids if eid in evidence_db]
    return f" {evid_sep} ".join(evidence_texts)


def build_examples(claim_dict, evidence_db, evid_sep: str):
    examples = []
    for claim_id, item in claim_dict.items():
        claim_text = item["claim_text"]
        text_b = build_text_b(item.get("evidences", []), evidence_db, evid_sep)
        label_str = item["claim_label"]
        examples.append(
            {
                "claim_id": claim_id,
                "text_a": claim_text,
                "text_b": text_b,
                "label": label2id[label_str],
            }
        )
    return examples


def build_noise_train_examples(
    train_claims,
    reranked_preds,
    evidence_db,
    evid_sep: str,
):
    examples = []
    skipped = 0
    for claim_id, item in train_claims.items():
        if claim_id not in reranked_preds:
            print(f"ERROR: {claim_id} missing in reranked predictions, skipping.")
            skipped += 1
            continue
        reranked_entry = reranked_preds[claim_id]
        if "evidences" not in reranked_entry:
            print(f"ERROR: {claim_id} has no 'evidences' in reranked predictions, skipping.")
            skipped += 1
            continue
        if "claim_text" not in item or "claim_label" not in item:
            print(f"ERROR: {claim_id} missing claim_text/claim_label in train-claims, skipping.")
            skipped += 1
            continue

        claim_text = item["claim_text"]
        text_b = build_text_b(reranked_entry["evidences"], evidence_db, evid_sep)
        label_str = item["claim_label"]
        examples.append(
            {
                "claim_id": claim_id,
                "text_a": claim_text,
                "text_b": text_b,
                "label": label2id[label_str],
            }
        )
    return examples, skipped


train_examples, train_skipped = build_noise_train_examples(
    train_claims, reranked_train, evidence_db, config.evid_sep
)
dev_examples = build_examples(dev_claims, evidence_db, config.evid_sep)

print(f"Train examples built: {len(train_examples)} (skipped {train_skipped})")
print("Sample example keys:", train_examples[0].keys())

Train examples built: 1228 (skipped 0)
Sample example keys: dict_keys(['claim_id', 'text_a', 'text_b', 'label'])


In [6]:
# =========================
# Cell 6: Tokenizer + Dataset
# =========================
tokenizer = AutoTokenizer.from_pretrained(config.model_name)


def tokenize_batch(batch):
    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        max_length=config.max_length,
        truncation="only_second",
    )


train_ds = Dataset.from_list(train_examples)
dev_ds = Dataset.from_list(dev_examples)

train_ds = train_ds.map(tokenize_batch, batched=True)
dev_ds = dev_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns(["claim_id", "text_a", "text_b"])
dev_ds = dev_ds.remove_columns(["claim_id", "text_a", "text_b"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [7]:
# =========================
# Cell 7: Model + Metrics
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=4,
    label2id=label2id,
    id2label=id2label,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# =========================
# Cell 8: Training Setup
# =========================
training_args = TrainingArguments(
    output_dir=config.output_dir,
    seed=config.seed,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    per_device_train_batch_size=config.train_batch_size,
    per_device_eval_batch_size=config.eval_batch_size,
    num_train_epochs=config.num_train_epochs,
    warmup_ratio=config.warmup_ratio,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
# =========================
# Cell 9: Train + Evaluate
# =========================
train_result = trainer.train()
print("Train result:", train_result.metrics)

dev_metrics = trainer.evaluate()
print("Dev metrics:", dev_metrics)

Epoch,Training Loss,Validation Loss,Accuracy
1,1.246308,1.260889,0.441558
2,1.217496,1.173949,0.506494
3,1.006441,1.164477,0.532468
4,0.882390,1.150203,0.545455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train result: {'train_runtime': 289.575, 'train_samples_per_second': 16.963, 'train_steps_per_second': 2.127, 'total_flos': 511883527808640.0, 'train_loss': 1.116160367990469, 'epoch': 4.0}


Dev metrics: {'eval_loss': 1.150202989578247, 'eval_accuracy': 0.5454545454545454, 'eval_runtime': 2.6436, 'eval_samples_per_second': 58.253, 'eval_steps_per_second': 7.565, 'epoch': 4.0}


In [10]:
# =========================
# Cell 10: Predict Function
# =========================
def predict_claim_label(
    model,
    tokenizer,
    claim_text: str,
    evidence_ids: list,
    evidence_db: dict,
) -> str:
    text_b = build_text_b(evidence_ids, evidence_db, config.evid_sep)
    enc = tokenizer(
        claim_text,
        text_b,
        truncation="only_second",
        max_length=config.max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**enc)
        pred_id = int(torch.argmax(outputs.logits, dim=-1).item())
    return id2label[pred_id]


def resolve_latest_checkpoint(output_dir: str) -> Path:
    root = Path(output_dir)
    checkpoints = sorted(
        root.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if checkpoints:
        return checkpoints[-1]
    return root


def compute_claim_classification_accuracy(predictions: dict, groundtruth: dict) -> float:
    """Same rule as eval.py: mean per-claim exact match on claim_label."""
    acc = []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue
        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0
        acc.append(instance_correct)
    return float(np.mean(acc)) if acc else 0.0

In [11]:
# =========================
# Cell 11: Reload Model + Compute A (eval.py style, reranked dev evidence)
# =========================
reranked_dev = load_json(Path(config.reranked_dev_path))
print(f"Reranked dev: {len(reranked_dev)}")

checkpoint_dir = resolve_latest_checkpoint(config.output_dir)
print(f"Reloading model from: {checkpoint_dir}")

reloaded_tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
reloaded_model.to("cuda" if torch.cuda.is_available() else "cpu")

dev_predictions = {}
dev_eval_skipped = 0
for claim_id, claim in sorted(dev_claims.items()):
    if claim_id not in reranked_dev:
        print(f"ERROR: {claim_id} missing in reranked dev predictions, skipping.")
        dev_eval_skipped += 1
        continue
    reranked_entry = reranked_dev[claim_id]
    if "evidences" not in reranked_entry:
        print(f"ERROR: {claim_id} has no 'evidences' in reranked dev predictions, skipping.")
        dev_eval_skipped += 1
        continue
    if "claim_text" not in claim:
        print(f"ERROR: {claim_id} missing claim_text in dev-claims, skipping.")
        dev_eval_skipped += 1
        continue

    evidence_ids = reranked_entry["evidences"]
    pred_label = predict_claim_label(
        reloaded_model,
        reloaded_tokenizer,
        claim["claim_text"],
        evidence_ids,
        evidence_db,
    )
    dev_predictions[claim_id] = {
        "claim_label": pred_label,
        "evidences": evidence_ids,
    }

print(f"Dev predictions built: {len(dev_predictions)} (skipped {dev_eval_skipped})")
claim_accuracy = compute_claim_classification_accuracy(dev_predictions, dev_claims)
print(f"Claim Classification Accuracy (A) = {claim_accuracy}")

Reranked dev: 154
Reloading model from: outputs/bert_claim_classifier_noise/checkpoint-616


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Dev predictions built: 154 (skipped 0)
Claim Classification Accuracy (A) = 0.5194805194805194
